In [5]:
#Import Libraries
import pandas as pd
import numpy as np

#Visualizations
import matplotlib.pyplot as plt
import seaborn as sns

#Import Machine Learning tests
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor

from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

#Import Dataset
df = pd.read_csv("../Project_european_airport_traffic_prediction_using_machine_learning_and_time_series_forecasting/flights.csv", header=0, index_col=0)

In [3]:
##Normalizing column headers

df.columns = (
    df.columns
    .str.lower()
    .str.replace('(', '_', regex=False)   # replace ( with _
    .str.replace('/', '_', regex=False)   # replace ( with _
    .str.replace(')', '', regex=False)    # remove )
    .str.replace(r'[^\w\s]', '', regex=True)  # remove other special chars
    .str.replace(' ', '_')                # spaces → underscore
    .str.replace('__', '_')
)
df.columns

Index(['month_num', 'month_mon', 'flt_date', 'apt_icao', 'apt_name',
       'state_name', 'flt_dep_1', 'flt_arr_1', 'flt_tot_1', 'flt_dep_ifr_2',
       'flt_arr_ifr_2', 'flt_tot_ifr_2', 'pivot_label'],
      dtype='object')

In [11]:
#split cols into different columns 
numerical_cols = df.select_dtypes(include=np.number)
print(numerical_cols)

#be careful because some categorical values my sip into numerical_cols (numbers that represent category)
categorical_cols = df.select_dtypes(exclude=np.number)
print(categorical_cols)

      MONTH_NUM  FLT_DEP_1  FLT_ARR_1  FLT_TOT_1  FLT_DEP_IFR_2  \
YEAR                                                              
2016          1          4          3          7            NaN   
2016          1        174        171        345          174.0   
2016          1         45         47         92           45.0   
2016          1          6          7         13            NaN   
2016          1          7          7         14            NaN   
...         ...        ...        ...        ...            ...   
2022          5         80         85        165            NaN   
2022          5         19         18         37            NaN   
2022          5         21         21         42           20.0   
2022          5         39         40         79            NaN   
2022          5         37         37         74            NaN   

      FLT_ARR_IFR_2  FLT_TOT_IFR_2  
YEAR                                
2016            NaN            NaN  
2016          161

In [14]:
df.head()

,MONTH_NUM,MONTH_MON,FLT_DATE,APT_ICAO,APT_NAME,STATE_NAME,FLT_DEP_1,FLT_ARR_1,FLT_TOT_1,FLT_DEP_IFR_2,FLT_ARR_IFR_2,FLT_TOT_IFR_2,Pivot Label
YEAR,,,,,,,,,,,,,
2016,1,JAN,2016-01-01T00:00:00Z,EBAW,Antwerp,Belgium,4,3,7,NaN,NaN,NaN,Antwerp (EBAW)
2016,1,JAN,2016-01-01T00:00:00Z,EBBR,Brussels,Belgium,174,171,345,174.0,161.0,335.0,Brussels (EBBR)
2016,1,JAN,2016-01-01T00:00:00Z,EBCI,Charleroi,Belgium,45,47,92,45.0,45.0,90.0,Charleroi (EBCI)
2016,1,JAN,2016-01-01T00:00:00Z,EBLG,Liège,Belgium,6,7,13,NaN,NaN,NaN,Liège (EBLG)
2016,1,JAN,2016-01-01T00:00:00Z,EBOS,Ostend-Bruges,Belgium,7,7,14,NaN,NaN,NaN,Ostend-Bruges (EBOS)


In [ ]:
#possibly drop the following columns as they include several NullValues: 

FLT_DEP_IFR_2    479785
FLT_ARR_IFR_2    479785
FLT_TOT_IFR_2    479785

#Explanation

# FLT_DEP_IFR_2	Number of IFR departures	278
# FLT_ARR_IFR_2	Number of IFR arrivals	241
# FLT_TOT_IFR_2	Number total IFR movements	519

In [12]:
#Explore all data
print("\n", "Head:","\n", df.head(5))
print("\n", "Tail:","\n", df.tail(5))
print("\n", "Null Values:","\n", df.isna().sum())
print("\n", "Info:","\n", df.info())
print("\n", "Info Memory Usage:","\n", df.info(memory_usage="deep"))
print("\n", "Describe:","\n", df.describe())
print("\n", "Describe All:","\n", df.describe(include="all"))
print("\n", "Shape:","\n", df.shape)
print("\n", "DTypes:","\n", df.dtypes)
print("\n", "Columns:","\n", df.columns)
print("\n", "Index:","\n", df.index)
print("\n", "Values:","\n", df.values)
print("\n", "Empty:","\n", df.empty)
#print("\n", "NDim:","\n", df.ndim)
print("\n", "Size:","\n", df.size)
#print("\n", "Memory True:","\n", df.memory_usage(deep=True))
print("\n", "Transpose:","\n", df.T)


 Head: 
       MONTH_NUM MONTH_MON              FLT_DATE APT_ICAO       APT_NAME  \
YEAR                                                                      
2016          1       JAN  2016-01-01T00:00:00Z     EBAW        Antwerp   
2016          1       JAN  2016-01-01T00:00:00Z     EBBR       Brussels   
2016          1       JAN  2016-01-01T00:00:00Z     EBCI      Charleroi   
2016          1       JAN  2016-01-01T00:00:00Z     EBLG          Liège   
2016          1       JAN  2016-01-01T00:00:00Z     EBOS  Ostend-Bruges   

     STATE_NAME  FLT_DEP_1  FLT_ARR_1  FLT_TOT_1  FLT_DEP_IFR_2  \
YEAR                                                              
2016    Belgium          4          3          7            NaN   
2016    Belgium        174        171        345          174.0   
2016    Belgium         45         47         92           45.0   
2016    Belgium          6          7         13            NaN   
2016    Belgium          7          7         14            Na

In [ ]:
#Column Description

# Column Name	Description	Example
# YEAR	Reference year	2014
# MONTH_NUM	Month (numeric)	1
# MONTH_MON	Month (3-letter code)	JAN
# FLT_DATE	Date of flight	01-Jan-2014
# APT_ICAO	ICAO 4-letter airport designator	EDDM
# APT_NAME	Airport name	Munich
# STATE_NAME	Name of the country in which the airport is located	Germany
# FLT_DEP_1	Number of IFR departures	278
# FLT_ARR_1	Number of IFR arrivals	241
# FLT_TOT_1	Number total IFR movements	519
# FLT_DEP_IFR_2	Number of IFR departures	278
# FLT_ARR_IFR_2	Number of IFR arrivals	241
# FLT_TOT_IFR_2	Number total IFR movements	519